# 05 - Construct robustness covariates from AlphaEarth embeddings

This notebook keeps AlphaEarth separate from the core covariate design. It aggregates AlphaEarth embeddings to the same analysis grid used by Notebook 01, freezes them in pre-treatment form where possible, and then reduces the 64 embedding dimensions into lower-dimensional robustness controls.

Production unit: one row per 1km x 1km grid cell.

Expected outputs:

- `data/intermediate/panel_robustness_covariates.parquet`
- `outputs/figures/05_alphaearth_scree_plot.png`
- `outputs/maps/05_alphaearth_cluster_map.png`
- `outputs/maps/05_alphaearth_pc1_map.png`


## Environment snapshot

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "project_config.json"
PROJECT_CONFIG = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}

RAW_DIR = Path(PROJECT_CONFIG.get("data_dirs", {}).get("raw", PROJECT_ROOT / "data" / "raw"))
INTERMEDIATE_DIR = Path(PROJECT_CONFIG.get("data_dirs", {}).get("intermediate", PROJECT_ROOT / "data" / "intermediate"))
TABLE_DIR = Path(PROJECT_CONFIG.get("output_dirs", {}).get("tables", PROJECT_ROOT / "outputs" / "tables"))
FIGURE_DIR = Path(PROJECT_CONFIG.get("output_dirs", {}).get("figures", PROJECT_ROOT / "outputs" / "figures"))
MAP_DIR = Path(PROJECT_CONFIG.get("output_dirs", {}).get("maps", PROJECT_ROOT / "outputs" / "maps"))

for path in [RAW_DIR, INTERMEDIATE_DIR, TABLE_DIR, FIGURE_DIR, MAP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## User configuration

AlphaEarth annual embeddings begin in 2017. Cells treated before or during the selected embedding year cannot receive a strictly pre-treatment AlphaEarth value unless earlier embeddings become available. By default this notebook uses 2017 only, which gives a clean baseline for cells not yet treated by 2017 and all never-treated cells.


In [ ]:
# --- Grid and treatment inputs ---
INTENDED_GRID_TAG = "1km"
ALLOW_LEGACY_GRID_FALLBACK = True
LEGACY_GRID_TAGS = ["0p080", ""]

# --- AlphaEarth source and timing ---
ALPHAEARTH_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
ALPHAEARTH_YEARS = [2017]
FREEZE_END_YEAR_NEVER_TREATED = 2017
N_ALPHA_BANDS = 64
ALPHA_BANDS = [f"A{i:02d}" for i in range(N_ALPHA_BANDS)]

# --- Earth Engine export controls ---
GEE_PROJECT = PROJECT_CONFIG.get("gee_project", "ee-cpedrazaj97")
RUN_GEE_EXPORTS = False
AUTO_START_EXPORT_TASKS = True
GEE_SCALE_ALPHAEARTH = 10
GEE_TILE_SCALE = 16
N_GRID_CHUNKS_1KM = 64
N_GRID_CHUNKS_LEGACY = 8
GRID_CHUNK_SEED = 42

# --- Dimensionality reduction controls ---
N_PCS = 10
N_CLUSTERS = 8
KMEANS_MAX_ITER = 100
KMEANS_RANDOM_SEED = 42

# --- Outputs ---
ROBUSTNESS_COVARIATE_PATH = INTERMEDIATE_DIR / "panel_robustness_covariates.parquet"
SCREE_PLOT_PATH = FIGURE_DIR / "05_alphaearth_scree_plot.png"
CLUSTER_MAP_PATH = MAP_DIR / "05_alphaearth_cluster_map.png"
PC1_MAP_PATH = MAP_DIR / "05_alphaearth_pc1_map.png"
CLUSTER_SUMMARY_PATH = TABLE_DIR / "05_alphaearth_cluster_summary.csv"


def tagged_path(stem, suffix, tag):
    if tag:
        return INTERMEDIATE_DIR / f"{stem}_{tag}{suffix}"
    return INTERMEDIATE_DIR / f"{stem}{suffix}"


def resolve_grid_artifacts(target_tag=INTENDED_GRID_TAG):
    candidates = [target_tag]
    if ALLOW_LEGACY_GRID_FALLBACK:
        candidates += [tag for tag in LEGACY_GRID_TAGS if tag not in candidates]
    for tag in candidates:
        panel_path = tagged_path("panel_treatment", ".parquet", tag)
        grid_path = tagged_path("grid_geometry", ".geojson", tag)
        if panel_path.exists() and grid_path.exists():
            return tag, panel_path, grid_path
    tried = [(str(tagged_path("panel_treatment", ".parquet", tag)), str(tagged_path("grid_geometry", ".geojson", tag))) for tag in candidates]
    raise FileNotFoundError(f"No matching treatment/grid artifacts found. Tried: {tried}")

GRID_TAG, PANEL_TREATMENT_PATH, GRID_GEOMETRY_PATH = resolve_grid_artifacts()
USING_INTENDED_GRID = GRID_TAG == INTENDED_GRID_TAG
N_GRID_CHUNKS = N_GRID_CHUNKS_1KM if GRID_TAG == "1km" else N_GRID_CHUNKS_LEGACY

GEE_EXPORT_FOLDER = f"gee_alphaearth_{GRID_TAG}"
GEE_EXPORT_PREFIX = f"alphaearth_{GRID_TAG}"
LOCAL_ALPHAEARTH_DIR = RAW_DIR / GEE_EXPORT_FOLDER
ALPHAEARTH_GLOB = f"{GEE_EXPORT_PREFIX}_year*_chunk*.csv"

print("Intended grid tag:", INTENDED_GRID_TAG)
print("Active grid tag:", GRID_TAG)
print("Using intended production grid:", USING_INTENDED_GRID)
if not USING_INTENDED_GRID:
    print("WARNING: running with legacy/development grid artifacts. Re-run after Notebook 01/02 produce 1km artifacts.")
print("Treatment panel:", PANEL_TREATMENT_PATH)
print("Grid geometry:", GRID_GEOMETRY_PATH)
print("AlphaEarth years:", ALPHAEARTH_YEARS)
print("GEE chunks:", N_GRID_CHUNKS)

## Verify active grid support

In [ ]:
grid_gdf = gpd.read_file(GRID_GEOMETRY_PATH)
if "cell_id" not in grid_gdf.columns:
    raise ValueError("Grid geometry is missing cell_id.")

grid_area_km2 = grid_gdf.to_crs("EPSG:3857").geometry.area / 1e6
print("Grid cells:", f"{len(grid_gdf):,}")
print("Median cell area km2:", round(float(grid_area_km2.median()), 3))
print("Cell area p05/p95 km2:", round(float(grid_area_km2.quantile(0.05)), 3), round(float(grid_area_km2.quantile(0.95)), 3))

if USING_INTENDED_GRID and not grid_area_km2.median().between(0.5, 1.5):
    raise ValueError("Active grid tag is 1km, but median cell area is not close to 1 km2.")
if not USING_INTENDED_GRID:
    print("Development note: this is not the intended 1km production grid.")

## Load treatment timing

The same freezing convention is used here: AlphaEarth years must be strictly before `first_treat_year` for treated cells. Never-treated cells use the configured baseline cutoff.


In [ ]:
panel = pd.read_parquet(PANEL_TREATMENT_PATH)
cell_timing = (
    panel[["cell_id", "first_treat_year", "ever_treated", "never_treated", "cell_lon", "cell_lat", "base_m2"]]
    .drop_duplicates("cell_id")
    .copy()
)
print(cell_timing.shape)
display(cell_timing.head())

## Optional Earth Engine export: aggregate AlphaEarth to grid cells

This block exports one row per grid cell per selected AlphaEarth year. For a 1km national grid, keep the chunked export pattern. The output CSVs should be downloaded from Google Drive into `LOCAL_ALPHAEARTH_DIR` before running the local PCA/clustering blocks.


In [ ]:
if RUN_GEE_EXPORTS:
    import ee
    import geemap

    try:
        ee.Initialize(project=GEE_PROJECT)
        print("Earth Engine initialized successfully.")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
        print("Earth Engine initialized successfully after authentication.")

    export_grid = grid_gdf[["cell_id", "geometry"]].copy()
    grid_ee = geemap.gdf_to_ee(export_grid)

    def add_grid_chunk(f):
        chunk = ee.Number(f.get("grid_random")).multiply(N_GRID_CHUNKS).floor().int()
        return f.set("grid_chunk", chunk)

    grid_ee = grid_ee.randomColumn("grid_random", seed=GRID_CHUNK_SEED).map(add_grid_chunk)

    tasks = {}
    for year in ALPHAEARTH_YEARS:
        alpha_img = (
            ee.ImageCollection(ALPHAEARTH_COLLECTION)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .mosaic()
            .select(ALPHA_BANDS)
        )
        renamed = alpha_img.rename([f"ae{year}_{b}" for b in ALPHA_BANDS])
        for chunk in range(N_GRID_CHUNKS):
            chunk_fc = grid_ee.filter(ee.Filter.eq("grid_chunk", chunk))
            reduced = renamed.reduceRegions(
                collection=chunk_fc,
                reducer=ee.Reducer.mean(),
                scale=GEE_SCALE_ALPHAEARTH,
                tileScale=GEE_TILE_SCALE,
            )
            description = f"{GEE_EXPORT_PREFIX}_year{year}_chunk{chunk:02d}"
            task = ee.batch.Export.table.toDrive(
                collection=reduced,
                description=description,
                folder=GEE_EXPORT_FOLDER,
                fileNamePrefix=description,
                fileFormat="CSV",
            )
            tasks[description] = task
            if AUTO_START_EXPORT_TASKS:
                task.start()
    print(("Started" if AUTO_START_EXPORT_TASKS else "Prepared"), len(tasks), "AlphaEarth export task(s).")

## Load downloaded AlphaEarth grid exports

In [ ]:
def read_alphaearth_exports(directory, glob_pattern):
    directory = Path(directory)
    files = sorted(directory.glob(glob_pattern)) if directory.exists() else []
    if not files:
        print("Missing AlphaEarth export files:", directory / glob_pattern)
        return None
    frames = []
    for path in files:
        df = pd.read_csv(path)
        if "cell_id" not in df.columns:
            raise ValueError(f"{path} does not contain cell_id")
        drop_cols = [c for c in ["system:index", ".geo", "id", "grid_random", "grid_chunk"] if c in df.columns]
        if drop_cols:
            df = df.drop(columns=drop_cols)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates("cell_id", keep="last")
    print("Loaded", len(files), "file(s) with", f"{len(out):,}", "unique cells.")
    return out

alpha_raw = read_alphaearth_exports(LOCAL_ALPHAEARTH_DIR, ALPHAEARTH_GLOB)
if alpha_raw is not None:
    display(alpha_raw.head())

## Freeze embeddings in pre-treatment form

The result is one 64-dimensional vector per grid cell. With the default 2017 baseline, cells already treated by 2017 are left missing because their embeddings are not strictly pre-treatment.


In [ ]:
def alpha_year_band_columns(df):
    pattern = re.compile(r"^ae(\d{4})_(A\d{2})$")
    out = []
    for col in df.columns:
        match = pattern.match(col)
        if match:
            out.append((int(match.group(1)), match.group(2), col))
    return out

if alpha_raw is None:
    alpha_frozen = None
    print("No AlphaEarth data loaded; skipping freeze/PCA/clustering until exports are available.")
else:
    alpha = cell_timing[["cell_id", "first_treat_year", "ever_treated", "never_treated"]].merge(alpha_raw, on="cell_id", how="left")
    year_band_cols = alpha_year_band_columns(alpha)
    frozen = alpha[["cell_id"]].copy()
    frozen["alphaearth_pre_years"] = 0

    for band in ALPHA_BANDS:
        cols = [(year, col) for year, b, col in year_band_cols if b == band]
        values = []
        counts = []
        for _, row in alpha.iterrows():
            row_values = []
            for year, col in cols:
                if row["ever_treated"] == 1:
                    keep = pd.notna(row["first_treat_year"]) and year < int(row["first_treat_year"])
                else:
                    keep = year <= FREEZE_END_YEAR_NEVER_TREATED
                if keep:
                    row_values.append(row.get(col, np.nan))
            series = pd.to_numeric(pd.Series(row_values), errors="coerce")
            values.append(float(series.mean()) if series.notna().any() else np.nan)
            counts.append(int(series.notna().sum()))
        frozen[f"alphaearth_pre_{band}"] = values
        frozen["alphaearth_pre_years"] = np.maximum(frozen["alphaearth_pre_years"], counts)

    alpha_frozen = cell_timing.merge(frozen, on="cell_id", how="left")
    print(alpha_frozen.shape)
    print("Cells with any pre-treatment AlphaEarth vector:", int(alpha_frozen[[f"alphaearth_pre_{b}" for b in ALPHA_BANDS]].notna().any(axis=1).sum()))
    display(alpha_frozen.head())

## PCA from grid-level embeddings

In [ ]:
def fit_pca_numpy(X, n_components):
    mean = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0)
    sd[sd == 0] = 1
    Z = (X - mean) / sd
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    scores = U[:, :n_components] * S[:n_components]
    eigenvalues = (S ** 2) / (len(X) - 1)
    explained = eigenvalues / eigenvalues.sum()
    return scores, Vt[:n_components], explained, mean, sd

if alpha_frozen is None:
    robustness_covariates = None
else:
    embedding_cols = [f"alphaearth_pre_{b}" for b in ALPHA_BANDS]
    valid = alpha_frozen[embedding_cols].notna().all(axis=1)
    X = alpha_frozen.loc[valid, embedding_cols].to_numpy(dtype=float)
    if len(X) < max(N_PCS + 1, N_CLUSTERS):
        robustness_covariates = None
        print("Not enough complete AlphaEarth vectors for PCA/clustering.")
    else:
        scores, loadings, explained, pca_mean, pca_sd = fit_pca_numpy(X, N_PCS)
        pc_cols = [f"alphaearth_pc{i+1:02d}" for i in range(N_PCS)]
        pc_df = pd.DataFrame(scores, columns=pc_cols, index=alpha_frozen.loc[valid].index)
        robustness_covariates = alpha_frozen[["cell_id", "cell_lon", "cell_lat", "first_treat_year", "ever_treated", "never_treated", "alphaearth_pre_years"]].copy()
        for col in pc_cols:
            robustness_covariates[col] = np.nan
        robustness_covariates.loc[valid, pc_cols] = pc_df

        scree = pd.DataFrame({"component": np.arange(1, len(explained) + 1), "explained_variance_share": explained})
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(scree["component"], scree["explained_variance_share"], marker="o")
        ax.set_xlabel("Principal component")
        ax.set_ylabel("Explained variance share")
        ax.set_title("AlphaEarth embedding PCA scree plot")
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(SCREE_PLOT_PATH, dpi=200)
        plt.show()
        print("Saved:", SCREE_PLOT_PATH)
        display(scree.head(15))

## Cluster embeddings into landscape types

In [ ]:
def kmeans_numpy(X, n_clusters, max_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=n_clusters, replace=False)].copy()
    labels = np.zeros(len(X), dtype=int)
    for _ in range(max_iter):
        distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        new_labels = distances.argmin(axis=1)
        new_centers = centers.copy()
        for k in range(n_clusters):
            if np.any(new_labels == k):
                new_centers[k] = X[new_labels == k].mean(axis=0)
        if np.array_equal(labels, new_labels):
            break
        labels = new_labels
        centers = new_centers
    return labels, centers

if robustness_covariates is None:
    print("Skipping clustering until PCA inputs are available.")
else:
    pc_cols = [c for c in robustness_covariates.columns if c.startswith("alphaearth_pc")]
    valid = robustness_covariates[pc_cols].notna().all(axis=1)
    X_cluster = robustness_covariates.loc[valid, pc_cols[:min(10, len(pc_cols))]].to_numpy(dtype=float)
    labels, centers = kmeans_numpy(X_cluster, N_CLUSTERS, KMEANS_MAX_ITER, KMEANS_RANDOM_SEED)
    robustness_covariates["alphaearth_cluster"] = pd.Series(pd.NA, index=robustness_covariates.index, dtype="Int64")
    robustness_covariates.loc[valid, "alphaearth_cluster"] = labels + 1
    print(robustness_covariates["alphaearth_cluster"].value_counts(dropna=False).sort_index())

## Compare clusters with intuitive geography

In [ ]:
if robustness_covariates is None or "alphaearth_cluster" not in robustness_covariates.columns:
    print("Skipping geography comparison until clusters are available.")
else:
    comparison = robustness_covariates.copy()
    core_path = INTERMEDIATE_DIR / "panel_core_covariates.parquet"
    if core_path.exists():
        core_cols = ["cell_id"] + [c for c in ["base_m2", "forest_loss_pre_mean_m2", "elevation_m_mean", "slope_deg_mean", "population_2000_sum"] if c in pd.read_parquet(core_path).columns]
        comparison = comparison.merge(pd.read_parquet(core_path, columns=core_cols), on="cell_id", how="left")

    summary_cols = [c for c in ["base_m2", "forest_loss_pre_mean_m2", "elevation_m_mean", "slope_deg_mean", "population_2000_sum", "alphaearth_pc01", "alphaearth_pc02"] if c in comparison.columns]
    cluster_summary = comparison.groupby("alphaearth_cluster", dropna=True)[summary_cols].mean().reset_index()
    cluster_summary.to_csv(CLUSTER_SUMMARY_PATH, index=False)
    print("Saved:", CLUSTER_SUMMARY_PATH)
    display(cluster_summary)

## Save robustness covariates and maps

In [ ]:
if robustness_covariates is None:
    print("No robustness covariate file written because AlphaEarth exports are not available yet.")
else:
    robustness_covariates.to_parquet(ROBUSTNESS_COVARIATE_PATH, index=False)
    print("Saved:", ROBUSTNESS_COVARIATE_PATH)

    map_gdf = grid_gdf[["cell_id", "geometry"]].merge(robustness_covariates, on="cell_id", how="left")

    if "alphaearth_cluster" in map_gdf.columns:
        fig, ax = plt.subplots(figsize=(8, 9))
        map_gdf.plot(column="alphaearth_cluster", categorical=True, legend=True, linewidth=0, ax=ax, missing_kwds={"color": "lightgrey"})
        ax.set_axis_off()
        ax.set_title("AlphaEarth embedding clusters")
        fig.tight_layout()
        fig.savefig(CLUSTER_MAP_PATH, dpi=200)
        plt.show()
        print("Saved:", CLUSTER_MAP_PATH)

    if "alphaearth_pc01" in map_gdf.columns:
        fig, ax = plt.subplots(figsize=(8, 9))
        map_gdf.plot(column="alphaearth_pc01", cmap="viridis", legend=True, linewidth=0, ax=ax, missing_kwds={"color": "lightgrey"})
        ax.set_axis_off()
        ax.set_title("AlphaEarth PC1")
        fig.tight_layout()
        fig.savefig(PC1_MAP_PATH, dpi=200)
        plt.show()
        print("Saved:", PC1_MAP_PATH)

## Final notes

The robustness design should remain separate from the main covariate table. AlphaEarth is useful for flexible landscape controls, but the main identification narrative should not depend on opaque embeddings before the core balance and robustness checks are understood.
